## DATA PREPROCESSING AND DATA CLEANING

In [ ]:
!!pip install -U torch torchvision torchaudio timm librosa soundfile sympy==1.11.1

In [ ]:
import zipfile

# Adjust name if your file name changes after re-upload
with zipfile.ZipFile("Dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("spine_dataset")

print("✅ Extracted successfully!")

,Open,High,Low,Close,Volume
0,122.800003,122.800003,119.820000,120.332497,30646000.0
1,121.237503,123.750000,120.625000,123.345001,24465208.0
2,123.312500,123.750000,122.000000,123.512497,21194656.0
3,123.750000,124.375000,122.949997,123.487503,19935544.0
4,123.737503,125.574997,123.250000,124.207497,21356352.0


In [ ]:
# ======== Install Dependencies ========
!pip install -U torch torchvision torchaudio timm librosa soundfile sympy==1.11.1

# ======== Imports & Device ========
import os, librosa, numpy as np, torch, timm, torch.nn as nn, torch.optim as optim, torchaudio, matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ======== Dataset Class ========
class AudioDataset(Dataset):
    def __init__(self, root_dir, sr=16000, duration=4):
        self.files, self.labels = [], []
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}
        self.sr, self.samples = sr, sr*duration
        for cls in self.classes:
            cls_path = os.path.join(root_dir, cls)
            for f in os.listdir(cls_path):
                if f.endswith(".wav"):
                    self.files.append(os.path.join(cls_path, f))
                    self.labels.append(self.class_to_idx[cls])
        self.mel = torchaudio.transforms.MelSpectrogram(sample_rate=sr, n_mels=128)
        self.db = torchaudio.transforms.AmplitudeToDB()
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        audio,_ = librosa.load(self.files[idx], sr=self.sr)
        if len(audio)<self.samples: audio = np.pad(audio,(0,self.samples-len(audio)))
        else: audio = audio[:self.samples]
        audio = torch.tensor(audio,dtype=torch.float32)
        mel = self.mel(audio)
        mel = self.db(mel)
        mel = mel.unsqueeze(0).repeat(3,1,1)  # 3 channels for AST
        return mel, self.labels[idx]

# ======== Paths & Hyperparameters ========
train_dir = "dataset/train"
val_dir = "dataset/test"
batch_size, num_epochs, learning_rate = 16, 6, 1e-4
num_classes = len(os.listdir(train_dir))

# ======== DataLoaders ========
train_dataset = AudioDataset(train_dir)
val_dataset = AudioDataset(val_dir)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# ======== Load Pretrained AST Model ========
model = timm.create_model("ast_base_patch16_224", pretrained=True)
model.head = nn.Linear(model.head.in_features, num_classes)
model = model.to(device)

# ======== Loss & Optimizer ========
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# ======== Training Loop ========
train_loss_history, val_loss_history = [], []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    train_loss = running_loss / len(train_loader.dataset)
    train_loss_history.append(train_loss)

    # Validation
    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    val_loss /= len(val_loader.dataset)
    val_loss_history.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# ======== Save Model ========
torch.save(model.state_dict(), "ast_audio_model.pth")
print("Model saved as ast_audio_model.pth")

# ======== Classification Report ========
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))

# ======== Plot Loss ========
plt.plot(train_loss_history, label="Train Loss")
plt.plot(val_loss_history, label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.title("Training vs Validation Loss")
plt.show()


,Open,High,Low,Close,Volume
4489,3150.000000,3155.350098,3128.550049,3144.699951,1793722.0
4490,3159.000000,3159.000000,3112.000000,3121.850098,1194289.0
4491,3105.000000,3160.000000,3105.000000,3157.300049,1587601.0
4492,3157.800049,3160.399902,3127.000000,3137.399902,1021913.0
4493,3170.100098,3178.000000,3155.000000,3161.699951,260949.0


In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Make sure model is in evaluation mode
model.eval()

# Lists to store true and predicted labels
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)
class_names = val_dataset.classes  # get class names from dataset

# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
plt.title("Confusion Matrix")
plt.show()


(4494, 5)

In [ ]:
from google.colab import files
uploaded = files.upload()
from PIL import Image
import torchvision.transforms as transforms
import torch

# Get uploaded image path
image_path = list(uploaded.keys())[0]
print("📷 Uploaded Image:", image_path)

# Define the same transform used during training
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# Prediction function
def predict_image(img_path, model, transform, class_names):
    model.eval()
    img = Image.open(img_path).convert('RGB')
    img = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(img)
        predicted = torch.argmax(output, 1).item()
    return class_names[predicted]

# Run prediction
predicted_class = predict_image(image_path, model, transform, val_dataset.classes)
print("✅ Predicted Class:", predicted_class)


In [ ]:
from google.colab import files

# Download your ViT model file
files.download('/content/vit_model.pth')